In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df_nykaa = pd.read_csv("Input/nykaa_campaign_data_with_nulls.csv")

print("Nykaa dataset loaded successfully!")
print("Shape:", df_nykaa.shape)

Nykaa dataset loaded successfully!
Shape: (55555, 16)


In [3]:
# Basic inspection of Nykaa dataset

print("Shape:", df_nykaa.shape)

print("\nColumns:")
print(df_nykaa.columns.tolist())

print("\nData Types:")
display(df_nykaa.dtypes)

print("\nFirst 5 Rows:")
display(df_nykaa.head())

Shape: (55555, 16)

Columns:
['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score', 'Customer_Segment', 'Date']

Data Types:


Campaign_ID             str
Campaign_Type           str
Target_Audience         str
Duration            float64
Channel_Used            str
Impressions         float64
Clicks              float64
Leads               float64
Conversions         float64
Revenue             float64
Acquisition_Cost    float64
ROI                 float64
Language                str
Engagement_Score    float64
Customer_Segment        str
Date                    str
dtype: object


First 5 Rows:


,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date
0,NY-CMP-1000,Social Media,College Students,21.0,"WhatsApp, YouTube",57804.0,6156.0,3616.0,2355.0,1867515.0,NaN,6.14,Hindi,20.98,College Students,29-04-2025
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18.0,YouTube,91801.0,3321.0,1971.0,1357.0,1046247.0,180.83,3.26,Hindi,7.24,College Students,06-04-2025
2,NY-CMP-1002,Influencer,Youth,23.0,"WhatsApp, Google, YouTube",15536.0,2182.0,952.0,755.0,197055.0,90.60,NaN,English,25.03,College Students,14-01-2025
3,NY-CMP-1003,Email,Working Women,18.0,"YouTube, Facebook, Instagram",88114.0,8413.0,2231.0,947.0,376906.0,249.07,0.60,Hindi,13.15,NaN,04-06-2025
4,NY-CMP-1004,Paid Ads,College Students,10.0,"Facebook, Instagram",96871.0,3743.0,2060.0,1258.0,518296.0,228.60,0.80,Hindi,7.29,Tier 2 City Customers,29-12-2024


In [4]:
# Check missing values in Nykaa dataset

missing_values = df_nykaa.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (missing_values / len(df_nykaa) * 100).round(2)
})

display(missing_summary[missing_summary["Missing_Count"] > 0])

,Missing_Count,Missing_Percentage
Campaign_Type,2859,5.15
Target_Audience,2761,4.97
Duration,2890,5.20
Channel_Used,2805,5.05
Impressions,2780,5.00
Clicks,2782,5.01
Leads,2779,5.00
Conversions,2666,4.80
Revenue,2812,5.06
Acquisition_Cost,2813,5.06


In [5]:
# Check duplicate records

print("Duplicate rows:", df_nykaa.duplicated().sum())

print("Duplicate Campaign_IDs:", df_nykaa["Campaign_ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Campaign_IDs: 0


In [6]:
# Handle missing Revenue and Acquisition_Cost

# Missing Revenue → 0
df_nykaa["Revenue"] = df_nykaa["Revenue"].fillna(0)

# Missing Acquisition_Cost → mean
acquisition_cost_mean = df_nykaa["Acquisition_Cost"].mean()
df_nykaa["Acquisition_Cost"] = df_nykaa["Acquisition_Cost"].fillna(acquisition_cost_mean)

print("Revenue missing values:", df_nykaa["Revenue"].isna().sum())
print("Acquisition_Cost missing values:", df_nykaa["Acquisition_Cost"].isna().sum())
print("Acquisition_Cost mean used:", acquisition_cost_mean)

Revenue missing values: 0
Acquisition_Cost missing values: 0
Acquisition_Cost mean used: 377.58721398505935


In [7]:
# Fill missing values in remaining numerical columns with mean
# ROI is intentionally excluded for now

other_numerical_columns = [
    "Duration",
    "Impressions",
    "Clicks",
    "Leads",
    "Conversions",
    "Engagement_Score"
]

for col in other_numerical_columns:
    df_nykaa[col] = df_nykaa[col].fillna(df_nykaa[col].mean())

print("Remaining numerical columns handled:")
print(other_numerical_columns)

print("\nMissing values after numerical imputation:")
display(df_nykaa[other_numerical_columns].isna().sum())

Remaining numerical columns handled:
['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Engagement_Score']

Missing values after numerical imputation:


Duration            0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Engagement_Score    0
dtype: int64

In [8]:
# Fill missing values in categorical columns with mode

categorical_columns = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Language",
    "Customer_Segment"
]

for col in categorical_columns:
    mode_value = df_nykaa[col].mode()[0]
    df_nykaa[col] = df_nykaa[col].fillna(mode_value)

print("Categorical columns handled using mode:")
print(categorical_columns)

print("\nMissing values after categorical imputation:")
display(df_nykaa[categorical_columns].isna().sum())

Categorical columns handled using mode:
['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']

Missing values after categorical imputation:


Campaign_Type       0
Target_Audience     0
Channel_Used        0
Language            0
Customer_Segment    0
dtype: int64

In [9]:
# Convert Date column to datetime

df_nykaa["Date"] = pd.to_datetime(
    df_nykaa["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Date datatype:", df_nykaa["Date"].dtype)
print("Missing dates after conversion:", df_nykaa["Date"].isna().sum())

display(df_nykaa[["Date"]].head())

Date datatype: datetime64[us]
Missing dates after conversion: 2758


,Date
0,2025-04-29
1,2025-04-06
2,2025-01-14
3,2025-06-04
4,2024-12-29


In [10]:
# Add Company Name
df_nykaa["Company_Name"] = "Nykaa"

# Calculate Profit
df_nykaa["Profit"] = (
    df_nykaa["Revenue"] -
    df_nykaa["Acquisition_Cost"]
)

# Calculate Profit Flag
df_nykaa["Profit_Flag"] = np.where(
    df_nykaa["Profit"] > 0,
    "Profit",
    "Loss"
)

# Calculate ROI
df_nykaa["Calculated_ROI"] = (
    df_nykaa["Profit"] /
    df_nykaa["Acquisition_Cost"]
)

print("Feature engineering completed successfully.")

display(
    df_nykaa[
        [
            "Company_Name",
            "Revenue",
            "Acquisition_Cost",
            "Profit",
            "Profit_Flag",
            "Calculated_ROI"
        ]
    ].head(10)
)

Feature engineering completed successfully.


,Company_Name,Revenue,Acquisition_Cost,Profit,Profit_Flag,Calculated_ROI
0,Nykaa,1867515.0,377.587214,1.867137e+06,Profit,4944.916945
1,Nykaa,1046247.0,180.830000,1.046066e+06,Profit,5784.804347
2,Nykaa,197055.0,90.600000,1.969644e+05,Profit,2174.000000
3,Nykaa,376906.0,249.070000,3.766569e+05,Profit,1512.253302
4,Nykaa,518296.0,228.600000,5.180674e+05,Profit,2266.261592
5,Nykaa,877020.0,137.790000,8.768822e+05,Profit,6363.903113
6,Nykaa,633402.0,210.170000,6.331918e+05,Profit,3012.760289
7,Nykaa,326525.0,147.290000,3.263777e+05,Profit,2215.885057
8,Nykaa,1058665.0,87.740000,1.058577e+06,Profit,12064.933440
9,Nykaa,603174.0,269.820000,6.029042e+05,Profit,2234.468090


In [11]:
# Validate remaining missing values after cleaning

missing_after_cleaning = df_nykaa.isnull().sum()

display(
    missing_after_cleaning[missing_after_cleaning > 0]
)

ROI     2825
Date    2758
dtype: int64

In [12]:
# Multi-label encoding for Channel_Used

channels = [
    "Google",
    "WhatsApp",
    "YouTube",
    "Email",
    "Instagram",
    "Facebook"
]

for channel in channels:
    df_nykaa[f"Channel_{channel}"] = (
        df_nykaa["Channel_Used"]
        .str.split(",")
        .apply(
            lambda x: int(channel in [item.strip() for item in x])
        )
    )

print("Multi-label encoding completed successfully.")

display(
    df_nykaa[
        [
            "Channel_Used",
            "Channel_Google",
            "Channel_WhatsApp",
            "Channel_YouTube",
            "Channel_Email",
            "Channel_Instagram",
            "Channel_Facebook"
        ]
    ].head(10)
)

Multi-label encoding completed successfully.


,Channel_Used,Channel_Google,Channel_WhatsApp,Channel_YouTube,Channel_Email,Channel_Instagram,Channel_Facebook
0,"WhatsApp, YouTube",0,1,1,0,0,0
1,YouTube,0,0,1,0,0,0
2,"WhatsApp, Google, YouTube",1,1,1,0,0,0
3,"YouTube, Facebook, Instagram",0,0,1,0,1,1
4,"Facebook, Instagram",0,0,0,0,1,1
5,"Email, Instagram",0,0,0,1,1,0
6,"YouTube, Google, WhatsApp",1,1,1,0,0,0
7,Instagram,0,0,0,0,1,0
8,"Facebook, WhatsApp, Email",0,1,0,1,0,1
9,Email,0,0,0,1,0,0


In [13]:
# Validate channel indicator totals

channel_columns = [
    "Channel_Google",
    "Channel_WhatsApp",
    "Channel_YouTube",
    "Channel_Email",
    "Channel_Instagram",
    "Channel_Facebook"
]

print("Number of campaigns using each channel:")

display(
    df_nykaa[channel_columns].sum().sort_values(ascending=False)
)

Number of campaigns using each channel:


Channel_Instagram    20396
Channel_WhatsApp     17604
Channel_Google       17598
Channel_YouTube      17586
Channel_Email        17524
Channel_Facebook     17399
dtype: int64

In [14]:
# Validate that every campaign has at least one encoded channel

channel_indicator_total = df_nykaa[channel_columns].sum(axis=1)

print("Campaigns with no encoded channel:",
      (channel_indicator_total == 0).sum())

print("Campaigns with one or more encoded channels:",
      (channel_indicator_total >= 1).sum())

print("Total campaigns:",
      len(df_nykaa))

Campaigns with no encoded channel: 0
Campaigns with one or more encoded channels: 55555
Total campaigns: 55555


In [15]:
# Save cleaned and feature-engineered Nykaa dataset

output_path = r"D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\Nykaa_Feature_Engineered.csv"

df_nykaa.to_csv(output_path, index=False)

print("Nykaa dataset saved successfully.")
print("File:", output_path)
print("Shape:", df_nykaa.shape)

Nykaa dataset saved successfully.
File: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\Nykaa_Feature_Engineered.csv
Shape: (55555, 26)
